# 검색 결과에 따라 움직이는 LangGraph 만들기

도서관 안내 그래프를 만든다. 답할 문장을 찾으면 인용하고, 못 찾으면 검색어를 한 번 고쳐 본다. 두 경우의 실행 기록을 나란히 읽는다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 패키지가 없는 커널에서는 새 코드 셀에 다음 명령을 실행한 뒤 예제로 이동하세요.

%pip install "langgraph>=1,<2" "langchain-core>=1,<2"

## 먼저 그래프가 처리할 일을 정하자

“연장”이라고 물으면 도서관 규정 하나를 찾아 안내한다. “책 더 빌리기”는 같은 뜻이지만 지금 검색 함수로는 찾지 못한다. “주차”는 애초에 자료가 없다. 세 질문은 같은 입구로 들어와 서로 다른 지점에서 끝나야 한다.

상태는 함수들이 함께 읽는 작업 기록이다. 원래 질문은 그대로 두고, 실제 검색에 넣은 표현과 찾은 근거를 별도 칸에 저장한다. 그래야 재검색 결과가 달라졌을 때 무엇을 바꿨는지 알 수 있다.

노드는 처리한 칸만 반환한다. 간선은 그다음 어떤 노드를 부를지 정한다. 처음에는 두 노드를 순서대로 연결한다. 조건 판단이 필요한 입력을 만났을 때 경로를 하나 더 만든다.

## 첫 연결에서 답 한 건 만들기

검색과 답변을 함수로 나눈 뒤 실제 StateGraph에 등록한다. 예제의 검색기는 비교할 표현을 눈으로 확인할 수 있게 작은 사전을 쓴다.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Desk(TypedDict, total=False):
    question: str
    query: str
    evidence: str | None
    attempts: int
    answer: str

policy = {"연장": "R2 · 대출은 한 번 연장할 수 있습니다."}
def search(state):
    query = state.get("query", state["question"]).strip()
    return {"query": query, "evidence": policy.get(query),
            "attempts": state.get("attempts", 0) + 1}

def respond(state):
    return {"answer": state["evidence"] or "규정을 확인했습니다."}

linear = StateGraph(Desk)
linear.add_node("search", search)
linear.add_node("respond", respond)
linear.add_edge(START, "search")
linear.add_edge("search", "respond")
linear.add_edge("respond", END)
first_app = linear.compile()
print(first_app.invoke({"question": "연장"})["answer"])

R2 · 대출은 한 번 연장할 수 있습니다.


START에서 들어간 입력은 search를 거치며 evidence와 attempts를 얻는다. respond가 answer만 반환해도 원래 question과 근거는 최종 상태에 남는다. compile은 연결을 준비하고 invoke가 질문 한 건을 실행한다.

## 근거가 없는데도 안내를 마친 이유

같은 그래프에 자료에 없는 질문을 넣는다. 모델의 표현을 고치기 전에 검색 노드의 결과부터 읽는다.

In [2]:
failed = first_app.invoke({"question": "주차"})
print(failed["evidence"], failed["answer"])

None 규정을 확인했습니다.


근거가 None인데도 respond로 갔다. 고정 간선이 모든 검색 결과를 같은 노드로 보냈고, 응답 함수의 기본 문장이 실패를 가렸다. 이 지점에서 다음 경로를 고르는 함수가 필요하다.

## 검색 뒤에 조건부 간선 끼워 넣기

경로 함수는 “answer”, “rewrite”, “hold” 중 하나를 고른다. 상태를 바꾸는 일은 선택된 노드가 맡는다. 새 그래프에서는 search의 다음 경로를 조건부 간선 한 곳에서만 정한다.

In [3]:
aliases = {"책 더 빌리기": "연장"}
def choose(state):
    if state["evidence"]:
        return "answer"
    if state["attempts"] < 2 and state["query"] in aliases:
        return "rewrite"
    return "hold"

def rewrite(state):
    return {"query": aliases[state["query"]]}

def hold(state):
    return {"answer": "근거 없음 · 담당자 확인"}

def build(route):
    graph = StateGraph(Desk)
    graph.add_node("search", search)
    graph.add_node("answer", respond)
    graph.add_node("rewrite", rewrite)
    graph.add_node("hold", hold)
    graph.add_edge(START, "search")
    graph.add_conditional_edges("search", route,
                                {"answer":"answer", "rewrite":"rewrite", "hold":"hold"})
    graph.add_edge("rewrite", "search")
    graph.add_edge("answer", END)
    graph.add_edge("hold", END)
    return graph

app = build(choose).compile()
result = app.invoke({"question":"책 더 빌리기"})
print(result["question"], result["query"], result["attempts"], result["answer"])

책 더 빌리기 연장 2 R2 · 대출은 한 번 연장할 수 있습니다.


질문 원문은 “책 더 빌리기”로 남고 두 번째 검색어만 “연장”으로 바뀌었다. rewrite가 answer를 직접 만들지 않고 search로 되돌아가므로 고친 검색어로 근거를 다시 찾는다.

## 실행 경로를 출력과 맞추기

```mermaid
flowchart TD
 S[질문] --> R[search]
 R -->|근거 있음| A[answer]
 R -->|근거 없음 · 별칭 있음 · 첫 시도| W[rewrite]
 R -->|그 밖의 경우| H[hold]
 W --> R
 A --> E[종료]
 H --> E
```

## 마지막 값과 실행 경로 구별하기

stream의 updates 모드로 어느 노드가 무엇을 돌려줬는지 순서대로 읽는다.

In [4]:
for update in app.stream({"question":"책 더 빌리기"}, stream_mode="updates"):
    for node, changed in update.items():
        print(node, changed)

search {'query': '책 더 빌리기', 'evidence': None, 'attempts': 1}
rewrite {'query': '연장'}
search {'query': '연장', 'evidence': 'R2 · 대출은 한 번 연장할 수 있습니다.', 'attempts': 2}
answer {'answer': 'R2 · 대출은 한 번 연장할 수 있습니다.'}


search가 두 번 나온다. 두 번째 search에서 evidence가 생긴 다음에만 answer가 나온다. 주차 문의를 대신 넣으면 별칭이 없어 첫 검색 직후 hold로 간다.

### 같은 검색어로 재시도하는 낭비 막기

별칭 자료에 “주차 → 주차”가 잘못 등록됐다. 기존 choose도 두 번 뒤에 멈추지만 두 번째 호출은 같은 일을 반복한다. 변환 전후가 같으면 첫 시도에서 보류하도록 경로 함수를 수정한다. 정상적인 “책 더 빌리기” 재검색은 살려 둔다.

In [5]:
aliases["주차"] = "주차"
print(app.invoke({"question":"주차"})["attempts"])

2


In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>def choose_changed(state):
    route = choose(state)
    if route == &quot;rewrite&quot; and aliases[state[&quot;query&quot;]] == state[&quot;query&quot;]:
        return &quot;hold&quot;
    return route

improved = build(choose_changed).compile()
print(improved.invoke({&quot;question&quot;:&quot;주차&quot;})[&quot;attempts&quot;])
print(improved.invoke({&quot;question&quot;:&quot;책 더 빌리기&quot;})[&quot;attempts&quot;])</code></pre><pre>1
2
</pre><p>첫 결과는 1, 두 번째는 2다. 별칭이 달라지는 경우에만 호출을 한 번 더 쓰고, 어떤 입력도 두 번을 넘기지 않는다.</p></details>

## 상태를 저장하면 다음 질문에 무엇이 남을까요?

체크포인터를 붙인 뒤 같은 대화에서 두 질문을 보낸다. 상태에 남겨야 할 대화 정보와 요청마다 초기화할 검색 정보를 구분해 보자.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
saved = build(choose).compile(checkpointer=InMemorySaver())
config = {"configurable":{"thread_id":"desk-a"}}
saved.invoke({"question":"연장"}, config)
next_turn = saved.invoke({"question":"주차"}, config)
print(next_turn["question"], next_turn["query"], next_turn["answer"])

주차 연장 R2 · 대출은 한 번 연장할 수 있습니다.


두 번째 질문은 주차인데 query에는 이전 질문의 “연장”이 남았다. 체크포인터는 직전 상태를 복원한다. search가 query를 우선하도록 짜여 있어 옛 근거로 새 문의에 답했다. 대화를 기억하는 기능만 추가해도 이와 같은 오류가 생길 수 있다.

### 대화를 유지하면서 검색 상태 초기화하기

같은 thread_id로 다시 요청한다. 처음 실행과 같은 횟수에서 출발하고, 이번 질문으로 검색하도록 입력을 만든다. 두 번째 대화 ID에 이 요청의 기록이 복사됐는지도 확인한다.

In [8]:
new_question = "주차"

In [9]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>fresh = {&quot;question&quot;:new_question, &quot;query&quot;:new_question, &quot;attempts&quot;:0}
turn = saved.invoke(fresh, config)
print(turn[&quot;query&quot;], turn[&quot;answer&quot;])
other = {&quot;configurable&quot;:{&quot;thread_id&quot;:&quot;desk-b&quot;}}
print(saved.get_state(other).values)</code></pre><pre>주차 근거 없음 · 담당자 확인
{}
</pre><p>검색 노드가 evidence를 새 결과로 덮어쓰고 hold가 보류 문장을 남긴다. 다른 thread_id의 빈 상태는 같은 사용자의 모든 질문을 하나의 전역 딕셔너리에 저장한 것이 아님을 보여 준다.</p></details>

## 질문에 답한 뒤 메시지 한 건을 바로잡기

체크포인터는 상태를 저장한다. 메시지 목록에 이번 답을 추가할지, 기존 답을 수정할지는 상태의 리듀서가 결정한다. 같은 메시지 ID를 유지한 채 안내 문구를 정정한다.

In [10]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
conversation = [HumanMessage(content="연장이 되나요?", id="q1"),
                AIMessage(content="두 번 가능합니다.", id="a1")]
corrected = add_messages(conversation,
                         [AIMessage(content="한 번 가능합니다.", id="a1")])
print([(m.id, m.content) for m in corrected])

[('q1', '연장이 되나요?'), ('a1', '한 번 가능합니다.')]


a1 자리에 정정한 문장 하나만 남았다. 질문이 새로 들어왔다면 새 메시지 ID를 써야 한다. 대화를 식별하는 thread_id와 그 안의 답 한 건을 식별하는 메시지 ID는 쓰임이 다르다.

### 잘못된 답과 새 답이 함께 남은 원인 찾기

정정문을 새 ID로 넣으면 메시지가 하나 더 생긴다. 같은 원본 대화에 두 방식을 각각 적용하고 항목 수와 마지막 두 ID를 비교한다.

In [11]:
new_reply = AIMessage(content="한 번 가능합니다.", id="a2")

In [12]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>appended = add_messages(conversation, [new_reply])
print(len(corrected), len(appended))
print([m.id for m in appended[-2:]])</code></pre><pre>2 3
[&#x27;a1&#x27;, &#x27;a2&#x27;]
</pre><p>새 메시지 a2를 붙이면 잘못된 a1도 남는다. 사용자가 하나의 답변이 정정됐다고 보게 하려면 원래 ID를 사용해야 한다.</p></details>

## 다른 입력에 적용하기

사전을 수업용 Retriever로 교체해 보자. 빈 후보와 도구 예외를 구별하고, 원문이 없을 때 답변 노드를 지나지 않는지 updates 기록으로 확인한다. 메시지 누적이 필요해지면 이어 읽기에서 add_messages를 열어 같은 메시지 ID를 수정할 때의 상태 변화를 실험한다.

직접 찾은 질문, 별칭으로 찾은 질문, 끝내 못 찾은 질문의 노드 순서·검색 횟수·근거를 기록한다. 이어 같은 대화에서 두 질문을 연달아 보내 이전 근거가 새 답에 붙지 않는지 확인한다.